# Notebook 8 — Room-segmentation evaluation (paper mean IoU + PQ)

**Stage GT file · runs last (after Notebook 7)**

> **Pipeline execution order: `6 -> 1 -> 2 -> 4 -> 7 -> 8`.**
> Notebook 8 scores each segmentation method's room-label map against the Notebook 7 ground
> truth. The **headline metric is the paper's point-based mean room IoU** (Albadri et al., ISPRS
> 2025, Eq. 6/7) with `p_i`/`g_i` matching (Eq. 3/4) — directly comparable to the published
> baseline. Panoptic Quality (SQ / RQ / PQ) is kept as a **secondary** number.

## Purpose
Per method: **mean room IoU** + over-/under-segmentation counts (primary), and SQ/RQ/PQ
(secondary). All scoring lives in `scan2bim.eval` (pure + unit-tested); this notebook is a thin
driver.

## Inputs
- `stage_gt/gt_room_labels.npy` (Notebook 7 — clean interior-area GT).
- `stage1_occupancy/wall_mask.npy` — the **shared wall scaffold** (every method + the GT exclude
  the *same* wall pixels, so nothing is scored on wall handling; Task 04/05).
- Each method reads its **own** stage (no aliasing — Task 03), via `scan2bim.load_method_labels`:
  - **Geometry**       ← `stage2_watershed/room_labels.npy`
  - **SAM**            ← `stage_sam_auto/room_labels.npy`  (pure automatic SAM, Method 2)
  - **Geometry + SAM** ← `stage4_sam_refined/room_labels_refined.npy`  (watershed refined by SAM)

## Outputs  (`{out_root}/stage_gt/`)
- `room_results.json` — per method: `mean_iou`, `per_room` (per-room IoU), `over_seg`,
  `under_seg`, `matched_pairs`, room counts (Task 07 aggregates this across areas).
- `pq_results.json` — per method `dict(SQ, RQ, PQ, TP, FP, FN)` (secondary).

## Assumptions
- All label maps share the Stage 1 grid (same `H x W`, same transform), so pixels correspond. A
  method whose stage is absent, or whose grid does not match the GT, is **skipped cleanly** ("not
  run") rather than crashing or being aliased to another method's array.

### Setup

In [1]:
# ============================== scan2bim setup (local) ==============================
import os
import json
import numpy as np
import scan2bim
from scan2bim import artifacts as A

CFG = scan2bim.load_config()
SHOW_DEBUG = True
STAGE_GT = 'stage_gt'
print('scan2bim', scan2bim.__version__, '| output root:', CFG.out_root)

scan2bim 1.0.0 | output root: c:\onestruction\scan2bim_out


### Step 1 — Load the GT, the shared wall scaffold, and each method's own labels
`scan2bim.load_method_labels` reads each method from its **own** stage (never aliasing two
methods). A method whose stage is missing (`None`) or whose grid does not match the GT is dropped
here with a printed reason, so the comparison only scores methods that actually ran on this grid.
The Stage-1 `wall_mask` is the shared scaffold passed to every score.

In [2]:
gt_dir = A.stage_dir(CFG.out_root, STAGE_GT)
gt_labels = A.load_npy(os.path.join(gt_dir, 'gt_room_labels.npy')).astype('int32')

# Shared wall scaffold: the Stage-1 occupancy wall pixels, excluded from BOTH sides so neither
# the GT nor any prediction is scored on wall handling (Task 04/05).
s1 = A.load_stage_dir(CFG.out_root, A.STAGE1)
wall_mask = A.load_npy(os.path.join(s1, A.WALLMASK_NPY)).astype(bool)
assert wall_mask.shape == gt_labels.shape, (wall_mask.shape, gt_labels.shape)

# Each method from its OWN stage (no aliasing). None -> stage not run; shape!=GT -> stale stage.
raw = scan2bim.load_method_labels(CFG.out_root)
methods = {}
for name, lab in raw.items():
    if lab is None:
        print('%-16s SKIP — stage not run' % name)
    elif lab.shape != gt_labels.shape:
        print('%-16s SKIP — grid %s != GT %s (stale stage; re-run on this grid)'
              % (name, lab.shape, gt_labels.shape))
    else:
        methods[name] = lab
        print('%-16s rooms=%3d  shape=%s'
              % (name, len([r for r in np.unique(lab) if r >= 1]), lab.shape))
print('GT rooms          :', len([r for r in np.unique(gt_labels) if r >= 1]),
      '| wall scaffold px:', int(wall_mask.sum()))
assert methods, 'No method produced labels on the GT grid — run the segmentation notebooks first.'

Geometry         rooms= 53  shape=(1606, 1618)
SAM              rooms= 37  shape=(1606, 1618)
Geometry + SAM   SKIP — grid (856, 1011) != GT (1606, 1618) (stale stage; re-run on this grid)
GT rooms          : 44 | wall scaffold px: 73675


### Step 2 — Scorers: paper mean IoU (primary) + PQ (secondary)
**Primary — `scan2bim.score_rooms`** (the paper's metric): per (pred, gt) pair `IoU =
|P∩G|/|P∪G|` (Eq. 6) over the shared grid, with the wall scaffold removed from both sides. Each
GT room is scored by its best-IoU prediction; the report is **mean IoU over GT rooms** (Eq. 7).
The `p_i`/`g_i` ratios (Eq. 3/4) flag **over-segmentation** (one GT room covered by ≥2
predictions) and **under-segmentation** (one prediction covering ≥2 GT rooms).

**Secondary — `compute_pq`**: standard panoptic quality. Greedy matching at `IoU > 0.5`; `SQ =
mean IoU over TPs`, `RQ = TP/(TP + 0.5·FP + 0.5·FN)`, `PQ = SQ·RQ`. Uses the same wall scaffold so
the two metrics are consistent.

In [3]:
# Primary scorer is scan2bim.score_rooms (paper Eq. 3/4/6/7) — imported, unit-tested, pure.
# Secondary PQ scorer below shares the wall scaffold so both metrics see the same valid cells.
def compute_pq(pred_labels, gt_labels, wall_mask=None):
    valid = np.ones(gt_labels.shape, bool) if wall_mask is None else ~np.asarray(wall_mask, bool)
    pred_ids = np.array([i for i in np.unique(pred_labels) if i >= 1])
    gt_ids = np.array([i for i in np.unique(gt_labels) if i >= 1])
    P, G = len(pred_ids), len(gt_ids)
    if P == 0 or G == 0:
        return dict(SQ=0.0, RQ=0.0, PQ=0.0, TP=0, FP=int(P), FN=int(G))

    pred_area = np.array([(valid & (pred_labels == v)).sum() for v in pred_ids], np.int64)
    gt_area = np.array([(valid & (gt_labels == v)).sum() for v in gt_ids], np.int64)

    inter = np.zeros((P, G), np.int64)
    both = valid & (pred_labels >= 1) & (gt_labels >= 1)
    if both.any():
        # pred_ids / gt_ids are sorted (np.unique), so searchsorted maps a label to its index.
        pr = np.searchsorted(pred_ids, pred_labels[both])
        gr = np.searchsorted(gt_ids, gt_labels[both])
        np.add.at(inter, (pr, gr), 1)

    union = pred_area[:, None] + gt_area[None, :] - inter
    iou = np.where(union > 0, inter / union, 0.0)

    pairs = sorted(((iou[i, j], i, j)
                    for i in range(P) for j in range(G) if iou[i, j] > 0.5),
                   reverse=True)
    pred_taken, gt_taken, matched = set(), set(), []
    for v, i, j in pairs:
        if i in pred_taken or j in gt_taken:
            continue
        pred_taken.add(i); gt_taken.add(j); matched.append(v)

    TP = len(matched); FP = P - TP; FN = G - TP
    SQ = float(np.mean(matched)) if TP else 0.0
    denom = TP + 0.5 * FP + 0.5 * FN
    RQ = TP / denom if denom else 0.0
    return dict(SQ=SQ, RQ=RQ, PQ=SQ * RQ, TP=int(TP), FP=int(FP), FN=int(FN))

### Step 3 — Score every method + print the tables
Primary table: the paper's **mean room IoU** + over-/under-seg. Secondary table: PQ. Both use the
shared wall scaffold.

In [4]:
room_results = {name: scan2bim.score_rooms(lab, gt_labels, wall_mask=wall_mask)
                for name, lab in methods.items()}
pq_results = {name: compute_pq(lab, gt_labels, wall_mask=wall_mask)
              for name, lab in methods.items()}

print('PRIMARY — paper mean room IoU (Eq. 6/7) + matching (Eq. 3/4)')
print('Method            meanIoU   over-seg  under-seg   #pred  #GT')
print('-' * 62)
for name, r in room_results.items():
    print('%-16s  %6.1f%%   %6d    %7d     %4d  %4d'
          % (name, 100 * r['mean_iou'], r['over_seg'], r['under_seg'],
             r['n_pred_rooms'], r['n_gt_rooms']))

print('\nSECONDARY — Panoptic Quality')
print('Method              SQ      RQ      PQ      TP   FP   FN')
print('-' * 58)
for name, r in pq_results.items():
    print('%-16s  %5.1f%%  %5.1f%%  %5.1f%%   %3d  %3d  %3d'
          % (name, 100 * r['SQ'], 100 * r['RQ'], 100 * r['PQ'],
             r['TP'], r['FP'], r['FN']))

PRIMARY — paper mean room IoU (Eq. 6/7) + matching (Eq. 3/4)
Method            meanIoU   over-seg  under-seg   #pred  #GT
--------------------------------------------------------------
Geometry            75.9%       10          0       53    44
SAM                 72.3%        1          0       37    44

SECONDARY — Panoptic Quality
Method              SQ      RQ      PQ      TP   FP   FN
----------------------------------------------------------
Geometry           86.7%   72.2%   62.6%    35   18    9
SAM                90.8%   84.0%   76.2%    34    3   10


### Step 4 — Save `room_results.json` (primary) + `pq_results.json` (secondary)
`room_results.json` carries the full per-room IoU / matched-pairs / over-/under-seg per method so
Task 07 can aggregate across areas.

In [5]:
out_dir = A.ensure_dir(gt_dir)
room_path = os.path.join(out_dir, 'room_results.json')
pq_path = os.path.join(out_dir, 'pq_results.json')
with open(room_path, 'w') as f:
    json.dump(room_results, f, indent=2)
with open(pq_path, 'w') as f:
    json.dump(pq_results, f, indent=2)
print('packaged ->', room_path)
print('packaged ->', pq_path)

packaged -> c:\onestruction\scan2bim_out\stage_gt\room_results.json
packaged -> c:\onestruction\scan2bim_out\stage_gt\pq_results.json
